# NBA PPM Researcher

Classical quantitative investigation of **`pts_per_min`** before further PPM model work.

**Notebook:** `models/nba/points/researcher.ipynb`  
**Spec:** `docs/superpowers/specs/2026-07-26-nba-ppm-researcher-design.md`

Independent of scoring discovery. No XGB/SHAP. No artifact saves.


In [ ]:
import warnings
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.feature_selection import mutual_info_regression

project_root = Path.cwd().resolve()
while not (project_root / "data").exists() and project_root != project_root.parent:
    project_root = project_root.parent
assert (project_root / "data").exists(), f"Could not find repo root from {Path.cwd()}"
sys.path.insert(0, str(project_root))
os.chdir(project_root)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("cwd:", Path.cwd())


## 1. Problem Definition

### Prediction problem
Pre-tip prediction of a player's **points per minute** in an upcoming NBA regular-season game (PPM → points prop path).

| Element | Definition |
|---------|------------|
| **Target** | `pts_per_min` (continuous) |
| **Observation unit** | Player-game after `(minutes >= 5) \| (starting == 1)` |
| **Prediction unit** | One forecast per player per upcoming game (pre-tip) |
| **Horizon** | Next game only |
| **Research metrics** | Spearman, mutual information, season rank stability |
| **Downstream modeling metric** | MAE on `pts_per_min` (see `models/README.md`) |

### Assumptions
1. Regular-season training parquets only (playoffs out of scope).
2. Row filter matches `model.ipynb`.
3. Predictive features are as-of prior games only (EWM / season_avg / lag / roll / context / market).
4. "Driver" means association, not proven causality.
5. `ContextFeatureEngineer(league="nba")` is part of the load path.
6. Research target remains `pts_per_min` (even if `pts ≈ pts_per_min * minutes`).

### Bias sources
- Playing-time / survivor bias from the minutes filter.
- Starter enrichment when `starting == 1` admits short outings.
- Season / rule-environment shifts across 2020–26.
- Role mix changes within players; missingness correlated with role.

### Leakage sources
- Same-game box / tracking / advanced rates used as pre-tip features.
- Target-derived contemporaneous efficiency.
- Using holdout `2025-26` to choose the feature shortlist.
- IDs treated as numeric predictors.


In [ ]:
from src.pipeline.features.context_features import ContextFeatureEngineer

SEASONS = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25", "2025-26"]
HOLDOUT_SEASON = "2025-26"
TARGET = "pts_per_min"
ID_COLS = [
    "game_id", "player_id", "team_id", "opp_team_id", "player_name",
    "game_date", "season_year", "matchup", "pos", "season",
]

PRIOR_TOKENS = ("_ewm_", "_season_avg", "_lag1", "_lag", "_roll", "_trend", "_rank", "_std", "_var")
CONTEXT_COLS = {
    "days_rest", "is_back_to_back", "games_played", "team_games_played",
    "games_played_last_7_days", "games_played_last_14_days", "min_sum_last_7_days",
    "starter_roll10_pct", "starting", "position_encoded",
}

CURRENT_PPM_FEATURES = [
    "base_pts_per_min_ewm_hl5",
    "base_pts_per_min_season_avg",
    "base_fga_per_min_ewm_hl10",
    "base_fg3a_per_min_ewm_hl10",
    "base_fta_per_min_ewm_hl10",
    "ts_pct_x_usg_pct",
    "adv_poss_ewm_hl5",
    "track_tchs_per_min_ewm_hl10",
    "track_cfga_per_min_ewm_hl10",
    "track_ufga_per_min_ewm_hl10",
    "opp_def_rating_ewm_hl10",
    "opp_pace_ewm_hl10",
    "team_pace_ewm_hl10",
]


def _is_market_col(col: str) -> bool:
    c = col.lower()
    if c.startswith("ou_") or "over_under" in c:
        return True
    return any(tok in c for tok in ("implied", "spread")) or "_line" in c


def lineage_for(col: str) -> str:
    c = col.lower()
    id_lower = {x.lower() for x in ID_COLS}
    if col == TARGET or col in {"minutes", "pts"} or col in ID_COLS or c in id_lower:
        return "excluded"
    if col == "ts_pct_x_usg_pct" or "_x_" in c:
        return "prior_player"
    if _is_market_col(col):
        return "market"
    if col in CONTEXT_COLS or c in CONTEXT_COLS or c.startswith("starter_"):
        return "context"
    if c.startswith("opp_"):
        return "opponent"
    if c.startswith("team_"):
        return "team"
    if any(tok in c for tok in PRIOR_TOKENS):
        return "prior_player"
    if c.startswith(("base_", "adv_", "track_")):
        return "same_game"
    return "same_game"


def split_feature_pools(columns):
    pools = {"predictive": [], "same_game": [], "excluded": []}
    for col in columns:
        lin = lineage_for(col)
        if lin == "excluded" or col == TARGET:
            pools["excluded"].append(col)
        elif lin == "same_game":
            pools["same_game"].append(col)
        elif lin in {"prior_player", "team", "opponent", "context", "market"}:
            pools["predictive"].append(col)
        else:
            pools["excluded"].append(col)
    return pools


frames = []
for season in SEASONS:
    path = Path(f"data/processed/{season}_Regular_Season_training_data.parquet")
    if not path.exists():
        print(f"WARNING: missing {path} — skipping")
        continue
    season_df = pd.read_parquet(path)
    season_df = ContextFeatureEngineer(league="nba").enrich(season_df)
    if "pos" in season_df.columns and "position_encoded" not in season_df.columns:
        season_df["position_encoded"] = season_df["pos"].map(
            {"PG": 1, "SG": 2, "SF": 3, "PF": 4, "C": 5}
        )
    frames.append(season_df)

df_raw = pd.concat(frames, ignore_index=True)
n_before = len(df_raw)
df = df_raw[(df_raw["minutes"] >= 5) | (df_raw["starting"] == 1)].copy()
df["game_date"] = pd.to_datetime(df["game_date"])
df = df.sort_values(["player_id", "game_date"]).reset_index(drop=True)

if {"adv_ts_pct_season_avg", "adv_usg_pct_season_avg"}.issubset(df.columns):
    df["ts_pct_x_usg_pct"] = df["adv_ts_pct_season_avg"] * df["adv_usg_pct_season_avg"]

print(f"Loaded {len(frames)} seasons | rows {n_before:,} → {len(df):,} after filter")
print(
    "Duplicate (game_id, player_id):",
    df.duplicated(subset=["game_id", "player_id"]).sum(),
)
print("Seasons:", sorted(df["season_year"].dropna().unique().tolist()))
print("Date range:", df["game_date"].min().date(), "→", df["game_date"].max().date())


In [ ]:
# Pool split + leakage audit
POOLS = split_feature_pools(df.columns)
pred_set, sg_set = set(POOLS["predictive"]), set(POOLS["same_game"])
assert pred_set.isdisjoint(sg_set), "Leakage: predictive ∩ same_game non-empty"
assert TARGET not in pred_set and TARGET not in sg_set

lineage_counts = (
    pd.Series({c: lineage_for(c) for c in df.columns})
    .value_counts()
    .rename_axis("lineage")
    .reset_index(name="n_cols")
)
print(lineage_counts.to_string(index=False))
print(
    f"\nPools — predictive={len(POOLS['predictive'])} | "
    f"same_game={len(POOLS['same_game'])} | excluded={len(POOLS['excluded'])}"
)
print("predictive ∩ same_game =", len(pred_set & sg_set))


## 2. Dataset Overview

Shape, season coverage, dtypes, and column families after enrich + filter.


In [ ]:
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
print("\nRows by season:")
print(df["season_year"].value_counts().sort_index().to_string())

print("\nDtype counts:")
print(df.dtypes.astype(str).value_counts().to_string())

families = {
    "base_": sum(c.startswith("base_") for c in df.columns),
    "adv_": sum(c.startswith("adv_") for c in df.columns),
    "track_": sum(c.startswith("track_") for c in df.columns),
    "opp_": sum(c.startswith("opp_") for c in df.columns),
    "team_": sum(c.startswith("team_") for c in df.columns),
    "context": sum(lineage_for(c) == "context" for c in df.columns),
    "same_game_pool": len(POOLS["same_game"]),
    "predictive_pool": len(POOLS["predictive"]),
}
print("\nColumn families:")
for k, v in families.items():
    print(f"  {k}: {v}")

df.head(5)


## 3. Data Quality Assessment

Per-column profile: dtype, missing %, uniqueness, cardinality, constants, outliers, invalids, memory. Flag suspicious columns.


In [ ]:
def cardinality_band(nunique: int, n: int) -> str:
    if nunique <= 1:
        return "const"
    frac = nunique / max(n, 1)
    if frac < 0.01 or nunique <= 10:
        return "low"
    if frac < 0.2:
        return "med"
    return "high"


def invalid_rate(s: pd.Series, col: str) -> float:
    if not pd.api.types.is_numeric_dtype(s):
        return 0.0
    x = pd.to_numeric(s, errors="coerce")
    valid = x.notna()
    if valid.sum() == 0:
        return 0.0
    c = col.lower()
    bad = pd.Series(False, index=x.index)
    if "pct" in c or c.endswith("_rate"):
        bad = valid & ~(((x >= 0) & (x <= 1.5)) | ((x >= 0) & (x <= 100)))
    elif "per_min" in c:
        bad = valid & (x < 0)
    elif c in {"minutes"}:
        bad = valid & ((x < 0) | (x > 60))
    elif "rating" in c:
        bad = valid & ((x < 0) | (x > 200))
    else:
        bad = valid & ~np.isfinite(x)
    return float(bad.sum() / valid.sum())


def outlier_rate_iqr(s: pd.Series) -> float:
    x = pd.to_numeric(s, errors="coerce").dropna()
    if len(x) < 20:
        return np.nan
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return 0.0
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return float(((x < lo) | (x > hi)).mean())


rows = []
n = len(df)
for col in df.columns:
    s = df[col]
    missing_pct = float(s.isna().mean() * 100)
    nunique = int(s.nunique(dropna=True))
    mode_pct = 0.0
    if nunique >= 1:
        mode_pct = float(s.value_counts(dropna=True, normalize=True).iloc[0] * 100)
    is_num = pd.api.types.is_numeric_dtype(s)
    mem = int(s.memory_usage(deep=True))
    rows.append({
        "column": col,
        "dtype": str(s.dtype),
        "missing_pct": round(missing_pct, 3),
        "nunique": nunique,
        "cardinality": cardinality_band(nunique, n),
        "is_constant": nunique <= 1,
        "is_near_constant": mode_pct >= 99.0 and nunique > 1,
        "mode_pct": round(mode_pct, 3),
        "outlier_pct": round(outlier_rate_iqr(s) * 100, 3) if is_num else np.nan,
        "invalid_pct": round(invalid_rate(s, col) * 100, 3) if is_num else np.nan,
        "memory_mb": round(mem / 1e6, 4),
        "lineage": lineage_for(col),
    })

quality = pd.DataFrame(rows).sort_values(["missing_pct", "column"], ascending=[False, True])
print(f"Frame memory: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print(f"Duplicate keys: {df.duplicated(subset=['game_id','player_id']).sum()}")
print(f"Constant cols: {quality['is_constant'].sum()} | Near-constant: {quality['is_near_constant'].sum()}")
quality.head(20)


In [ ]:
# Suspicious column flags
flags = []
for _, r in quality.iterrows():
    reasons = []
    if r["missing_pct"] >= 30:
        reasons.append("high_missing")
    if r["is_constant"]:
        reasons.append("constant")
    if r["is_near_constant"]:
        reasons.append("near_constant")
    if r["lineage"] == "same_game":
        reasons.append("leakage_risk_if_pre_tip")
    if r["column"] in ID_COLS:
        reasons.append("id_like")
    if pd.notna(r["invalid_pct"]) and r["invalid_pct"] > 1:
        reasons.append("broken_scale")
    if reasons:
        flags.append({
            "column": r["column"],
            "lineage": r["lineage"],
            "flags": ",".join(reasons),
            "missing_pct": r["missing_pct"],
            "nunique": r["nunique"],
        })

suspicious = pd.DataFrame(flags)
print(f"Flagged columns: {len(suspicious)}")
suspicious.head(40)


In [ ]:
# Missingness visualization
miss = quality.loc[quality["missing_pct"] > 0].sort_values("missing_pct", ascending=False).head(40)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if len(miss):
    axes[0].barh(miss["column"][::-1], miss["missing_pct"][::-1], color="steelblue")
    axes[0].set_xlabel("Missing %")
    axes[0].set_title("Top missing columns")
else:
    axes[0].text(0.5, 0.5, "No missing values", ha="center", va="center")
    axes[0].set_axis_off()

fam_miss = (
    quality.assign(
        family=lambda d: np.select(
            [
                d["column"].str.startswith("base_"),
                d["column"].str.startswith("adv_"),
                d["column"].str.startswith("track_"),
                d["column"].str.startswith("opp_"),
                d["column"].str.startswith("team_"),
                d["lineage"].eq("context"),
            ],
            ["base", "adv", "track", "opp", "team", "context"],
            default="other",
        )
    )
    .groupby("family")["missing_pct"]
    .mean()
    .sort_values(ascending=False)
)
axes[1].bar(fam_miss.index, fam_miss.values, color="darkorange")
axes[1].set_ylabel("Mean missing %")
axes[1].set_title("Mean missingness by family")
plt.tight_layout()
plt.show()


## 4. Target Analysis

Distribution, moments, outliers, temporal structure, autocorrelation, and modeling implications for `pts_per_min`.


In [ ]:
y = df[TARGET].dropna()
moments = {
    "n": int(y.shape[0]),
    "mean": float(y.mean()),
    "median": float(y.median()),
    "variance": float(y.var()),
    "std": float(y.std()),
    "skew": float(y.skew()),
    "kurtosis": float(y.kurtosis()),
    "min": float(y.min()),
    "p01": float(y.quantile(0.01)),
    "p99": float(y.quantile(0.99)),
    "max": float(y.max()),
}
q1, q3 = y.quantile(0.25), y.quantile(0.75)
iqr = q3 - q1
outlier_pct = float((((y < q1 - 1.5 * iqr) | (y > q3 + 1.5 * iqr)).mean()) * 100)
moments["outlier_pct_iqr"] = outlier_pct
pd.Series(moments).round(4)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(y, bins=60, color="steelblue", edgecolor="white")
axes[0].set_title("Histogram: pts_per_min")
axes[0].set_xlabel("pts_per_min")
axes[0].set_ylabel("Count")

axes[1].boxplot(y, vert=True, patch_artist=True,
                boxprops=dict(facecolor="lightsteelblue"))
axes[1].set_title("Boxplot: pts_per_min")
axes[1].set_ylabel("pts_per_min")

stats.probplot(y.sample(min(len(y), 5000), random_state=RANDOM_SEED), dist="norm", plot=axes[2])
axes[2].set_title("QQ plot vs Normal")
plt.tight_layout()
plt.show()


In [ ]:
# Seasonality / trend
by_season = df.groupby("season_year")[TARGET].agg(["count", "mean", "median", "std"])
print("By season:")
display(by_season)

df["_month"] = df["game_date"].dt.month
by_month = df.groupby("_month")[TARGET].agg(["count", "mean", "median"])
print("By calendar month:")
display(by_month)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
by_season["mean"].plot(ax=axes[0], marker="o", title="Mean pts_per_min by season")
axes[0].set_ylabel("mean pts_per_min")
by_month["mean"].plot(ax=axes[1], marker="o", title="Mean pts_per_min by month")
axes[1].set_xlabel("month")
axes[1].set_ylabel("mean pts_per_min")
plt.tight_layout()
plt.show()


In [ ]:
# Player-level lag-1 autocorrelation (panel, not a single AR series)
tmp = df[["player_id", "game_date", TARGET]].dropna().sort_values(["player_id", "game_date"])
tmp["y_lag1"] = tmp.groupby("player_id")[TARGET].shift(1)

def _corr(g):
    if g["y_lag1"].notna().sum() < 10:
        return np.nan
    return g[TARGET].corr(g["y_lag1"])

player_ac1 = tmp.groupby("player_id", group_keys=False).apply(_corr).dropna()
print(f"Players with AC1 estimate: {len(player_ac1)}")
print(f"Mean lag-1 corr: {player_ac1.mean():.4f} | median: {player_ac1.median():.4f}")

daily = df.groupby("game_date")[TARGET].mean()
fig, ax = plt.subplots(figsize=(12, 3))
daily.rolling(14, min_periods=5).mean().plot(ax=ax, title="League mean pts_per_min (14-day smooth)")
ax.set_ylabel("pts_per_min")
plt.tight_layout()
plt.show()


### Modeling implications (target)

- **Skew / heavy right tail / IQR outliers:** point prediction with MAE is appropriate; quantile models (p10/p50/p90) in the trainer handle asymmetry better than Gaussian MSE alone.
- **Non-normal QQ:** do not rely on normality-based intervals without calibration checks.
- **Positive player-level lag-1 autocorrelation:** prior form (EWM / lag1 / season_avg) is justified; residual clustering within players is expected.
- **Season / month mean shifts:** treat as panel with regime drift — hold out `2025-26` and prefer season-aware validation over random splits.
- **Stationarity:** this is **not** a single time series; league-mean drift is mild diagnostic only. Model player-games with as-of features, not undifferentiated ARIMA on the stacked frame.


## 5. Feature Exploration

Prior-only predictive pool by lineage. Same-game stats are **anatomy only** — leakage if used pre-tip.


In [ ]:
pred_cols = [c for c in POOLS["predictive"] if pd.api.types.is_numeric_dtype(df[c])]
sg_cols = [c for c in POOLS["same_game"] if pd.api.types.is_numeric_dtype(df[c])]

lin_pred = pd.Series({c: lineage_for(c) for c in pred_cols}).value_counts()
print("Predictive numeric columns by lineage:")
print(lin_pred.to_string())
print(f"\nSame-game numeric (anatomy only): {len(sg_cols)}")

sample_feats = [c for c in pred_cols if any(k in c for k in (
    "pts_per_min_ewm", "fga_per_min_ewm", "usg_pct", "tchs_per_min",
    "opp_def_rating", "team_pace", "days_rest", "poss_ewm"
))][:12]
if sample_feats:
    display(df[sample_feats].describe().T)


In [ ]:
# SAME-GAME anatomy — NOT model features / leakage if used pre-tip
print("=" * 72)
print("SAME-GAME ANATOMY (Spearman vs pts_per_min) — LEAKAGE IF USED PRE-TIP")
print("=" * 72)

def spearman_table(frame, cols, target=TARGET, top_n=15):
    yv = frame[target]
    rows = []
    for c in cols:
        pair = pd.concat([frame[c], yv], axis=1).dropna()
        if len(pair) < 100:
            continue
        rho, p = stats.spearmanr(pair.iloc[:, 0], pair.iloc[:, 1])
        rows.append({
            "feature": c,
            "spearman": rho,
            "p_value": p,
            "n": len(pair),
            "lineage": lineage_for(c),
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["abs_spearman"] = out["spearman"].abs()
    return out.sort_values("abs_spearman", ascending=False).drop(columns="abs_spearman").head(top_n)

anatomy = spearman_table(df, sg_cols, top_n=15)
display(anatomy)


## 6. Relationship Analysis

Prior-only Spearman + mutual information vs `pts_per_min`, fit on **pre-holdout** rows only for shortlist decisions. Holdout is not used here.


In [ ]:
df_rank = df[df["season_year"] != HOLDOUT_SEASON].copy()
print(f"Ranking frame (pre-holdout): {len(df_rank):,} rows | holdout excluded: {HOLDOUT_SEASON}")

# Documented MI rule: median-fill numeric predictors after row filter
X_pred = df_rank[pred_cols].copy()
medians = X_pred.median(numeric_only=True)
X_filled = X_pred.fillna(medians)
y_rank = df_rank[TARGET]

keep = []
for c in pred_cols:
    s = X_filled[c]
    if s.notna().sum() == 0:
        continue
    if s.nunique(dropna=True) <= 1:
        continue
    keep.append(c)
X_filled = X_filled[keep]

n_mi = min(len(X_filled), 20000)
rng = np.random.default_rng(RANDOM_SEED)
idx = rng.choice(len(X_filled), size=n_mi, replace=False)
X_mi = X_filled.iloc[idx]
y_mi = y_rank.iloc[idx]

print(f"MI on {n_mi:,} rows × {X_mi.shape[1]} features (median-filled)")

mi_vals = mutual_info_regression(X_mi, y_mi, random_state=RANDOM_SEED)
mi_map = dict(zip(X_mi.columns, mi_vals))

rows = []
for c in keep:
    pair = pd.concat([df_rank[c], y_rank], axis=1).dropna()
    if len(pair) < 100:
        continue
    rho, p = stats.spearmanr(pair.iloc[:, 0], pair.iloc[:, 1])
    rows.append({
        "feature": c,
        "spearman": rho,
        "abs_spearman": abs(rho),
        "mi": mi_map.get(c, np.nan),
        "p_value": p,
        "n": len(pair),
        "lineage": lineage_for(c),
        "missing_pct": float(df_rank[c].isna().mean() * 100),
    })

univariate = pd.DataFrame(rows).sort_values(["abs_spearman", "mi"], ascending=False)
print(f"Ranked features: {len(univariate)}")
univariate.head(30)


In [ ]:
# Collinearity among top candidates
TOP_N = 25
top_feats = univariate.head(TOP_N)["feature"].tolist()
corr = df_rank[top_feats].corr(method="spearman")
high_pairs = []
cols = list(corr.columns)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = corr.iloc[i, j]
        if abs(r) >= 0.90:
            high_pairs.append({"a": cols[i], "b": cols[j], "spearman": r})
high_corr = pd.DataFrame(high_pairs)
if len(high_corr):
    high_corr = high_corr.sort_values("spearman", key=np.abs, ascending=False)
print(f"Pairs |rho|>=0.90 among top {TOP_N}: {len(high_corr)}")
display(high_corr.head(20))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap="vlag", center=0, ax=ax, xticklabels=True, yticklabels=True)
ax.set_title(f"Spearman corr — top {TOP_N} prior-only features")
plt.tight_layout()
plt.show()


In [ ]:
# Scatter / hex for top drivers
top6 = univariate.head(6)["feature"].tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
sample = df_rank.sample(min(len(df_rank), 8000), random_state=RANDOM_SEED)
for ax, feat in zip(axes, top6):
    ax.hexbin(sample[feat], sample[TARGET], gridsize=40, cmap="Blues", mincnt=1)
    ax.set_xlabel(feat)
    ax.set_ylabel(TARGET)
    ax.set_title(feat[:40])
plt.suptitle("Top prior-only drivers vs pts_per_min (hexbin)")
plt.tight_layout()
plt.show()


## 7. Segmentation

How `pts_per_min` varies by role, minutes, scoring tier, and position.


In [ ]:
def segment_summary(frame, key):
    return (
        frame.groupby(key)[TARGET]
        .agg(n="count", mean="mean", median="median", std="std")
        .sort_index()
    )

print("By starting:")
display(segment_summary(df, "starting"))

df["minutes_tier"] = pd.cut(
    df["minutes"],
    bins=[-np.inf, 15, 25, 35, np.inf],
    labels=["<15", "15-25", "25-35", "35+"],
)
print("By minutes tier:")
display(segment_summary(df, "minutes_tier"))

df["ppm_tier"] = pd.cut(
    df[TARGET],
    bins=[-np.inf, 0.3, 0.5, 0.7, np.inf],
    labels=["<0.3", "0.3-0.5", "0.5-0.7", "0.7+"],
)
print("By PPM tier (target bins — descriptive only):")
display(df["ppm_tier"].value_counts().sort_index())

if "pos" in df.columns:
    print("By position:")
    display(segment_summary(df, "pos"))
elif "position_encoded" in df.columns:
    print("By position_encoded:")
    display(segment_summary(df, "position_encoded"))


## 8. Temporal Analysis

Season means, feature-rank stability across seasons, and **descriptive** holdout drift (not used for shortlisting).


In [ ]:
train_seasons = [s for s in SEASONS if s != HOLDOUT_SEASON and s in set(df["season_year"])]

def season_feature_ranks(frame, features, top_n=30):
    ranks = {}
    for season in train_seasons:
        sub = frame[frame["season_year"] == season]
        scores = {}
        for c in features:
            pair = pd.concat([sub[c], sub[TARGET]], axis=1).dropna()
            if len(pair) < 80:
                continue
            rho, _ = stats.spearmanr(pair.iloc[:, 0], pair.iloc[:, 1])
            scores[c] = abs(rho)
        if not scores:
            continue
        order = pd.Series(scores).sort_values(ascending=False).head(top_n)
        ranks[season] = {f: i for i, f in enumerate(order.index, start=1)}
    return ranks


stab_features = univariate.head(40)["feature"].tolist()
ranks = season_feature_ranks(df_rank, stab_features, top_n=30)

stability_rows = []
seasons_present = list(ranks.keys())
for i, s1 in enumerate(seasons_present):
    for s2 in seasons_present[i + 1 :]:
        common = set(ranks[s1]) & set(ranks[s2])
        if len(common) < 5:
            continue
        r1 = [ranks[s1][f] for f in common]
        r2 = [ranks[s2][f] for f in common]
        rho, _ = stats.spearmanr(r1, r2)
        stability_rows.append({
            "season_a": s1,
            "season_b": s2,
            "rank_spearman": rho,
            "n_common": len(common),
        })

stability = pd.DataFrame(stability_rows)
print("Cross-season rank stability (Spearman of feature ranks):")
display(stability)
if len(stability):
    print(f"Mean rank stability: {stability['rank_spearman'].mean():.4f}")


In [ ]:
# Holdout drift — DIAGNOSTIC ONLY (do not choose shortlist from this)
df_hold = df[df["season_year"] == HOLDOUT_SEASON]
print("=" * 72)
print(f"HOLDOUT DRIFT DIAGNOSTIC ONLY — {HOLDOUT_SEASON} (not for shortlist decisions)")
print("=" * 72)

drift_rows = [{
    "metric": TARGET,
    "train_mean": df_rank[TARGET].mean(),
    "holdout_mean": df_hold[TARGET].mean() if len(df_hold) else np.nan,
    "train_std": df_rank[TARGET].std(),
    "holdout_std": df_hold[TARGET].std() if len(df_hold) else np.nan,
}]
for feat in univariate.head(10)["feature"]:
    drift_rows.append({
        "metric": feat,
        "train_mean": df_rank[feat].mean(),
        "holdout_mean": df_hold[feat].mean() if len(df_hold) else np.nan,
        "train_std": df_rank[feat].std(),
        "holdout_std": df_hold[feat].std() if len(df_hold) else np.nan,
    })
drift = pd.DataFrame(drift_rows)
drift["mean_delta"] = drift["holdout_mean"] - drift["train_mean"]
display(drift)


## 9. Feature Engineering Ideas

Hypotheses only — **not implemented** in this notebook.

1. **Keep / stress-test** recent PPM form (`base_pts_per_min_ewm_hl5`, season_avg) — usually top univariate signal.
2. **Volume × efficiency** — retain `ts_pct_x_usg_pct`; try `fga_ewm × ts_ewm` and contested vs uncontested share (`cfga`/`ufga` ratio).
3. **Touches → shots** — `tchs_per_min_ewm` interacting with `fga_per_min_ewm` (shot creation conversion).
4. **Matchup** — opponent DEF_RATING × player usage; pace mismatch (`team_pace - opp_pace`).
5. **Rest** — if `days_rest` / B2B present after enrich, interact with usage for stars vs bench.
6. **Stability prune** — drop near-duplicate EWM horizons that are |ρ|≥0.90 with a shorter half-life already in the model.
7. **Gaps to ingest later** (out of scope here): primary defender, injury usage redistribution, line movement.


## 10. Modeling Readiness

Leakage audit, recommended shortlist vs avoid list, and reminders for `model.ipynb`.


In [ ]:
print("Leakage audit")
print("  predictive ∩ same_game:", len(set(POOLS["predictive"]) & set(POOLS["same_game"])))
print("  TARGET in predictive:", TARGET in POOLS["predictive"])
print("  Shortlist fit seasons: pre-holdout only —", [s for s in train_seasons])

rec = univariate.copy()
rec = rec.merge(
    quality[["column", "is_near_constant", "is_constant", "missing_pct"]],
    left_on="feature",
    right_on="column",
    how="left",
)
miss_col = "missing_pct_x" if "missing_pct_x" in rec.columns else "missing_pct"
rec = rec[
    (~rec["is_constant"].fillna(False))
    & (~rec["is_near_constant"].fillna(False))
    & (rec[miss_col].fillna(100) < 30)
]
shortlist = rec.head(25)[["feature", "spearman", "mi", "lineage", miss_col]].rename(
    columns={miss_col: "missing_pct"}
)
print("\nRecommended shortlist (top 25 prior-only, quality-filtered):")
display(shortlist)

avoid = suspicious.loc[
    suspicious["flags"].str.contains("constant|near_constant|broken_scale|high_missing", na=False),
    ["column", "flags", "missing_pct"],
].head(40)
print("\nAvoid / fix list (sample):")
display(avoid)

print("\nCurrent model.ipynb PPM_FEATURES coverage in shortlist top 40:")
top40 = set(univariate.head(40)["feature"])
for f in CURRENT_PPM_FEATURES:
    if f not in df.columns:
        status = "missing col"
    elif f in top40:
        status = "IN top40"
    else:
        status = "outside top40"
    print(f"  {f}: {status}")

print()
print("Reminders for model.ipynb:")
print("- Primary metric: MAE on pts_per_min; Wilcoxon vs naive on holdout.")
print("- Holdout 2025-26: evaluate once; no early stopping / feature selection on it.")
print("- Do not add same_game columns to PPM_FEATURES.")
print("- Prefer date-safe walk-forward within train seasons.")


## 11. Conclusions

### Executive findings
1. **Data:** Multi-season player-game panel after minutes/starter filter is usable; inspect high-missing tracking/context columns before trusting them.
2. **Target:** `pts_per_min` is right-skewed with outliers and positive within-player autocorrelation → MAE + quantiles + prior-form features are justified.
3. **Drivers:** Prior scoring form, shot volume/usage, touches, and pace/defense context dominate leakage-safe associations; same-game FGA/TS/USG explain the target mechanically but are **not** pre-tip features.
4. **Stability:** Use cross-season rank stability to prefer features that stay high across years; prune collinear EWM duplicates.
5. **Next for `model.ipynb`:** Compare `PPM_FEATURES` to the shortlist above — keep core form/volume/tracking/matchup signals, drop unstable or redundant horizons, only add shortlist features that are prior-safe and not |ρ|≥0.90 clones.

### Success criteria checklist
- [x] Problem defined (target, units, horizon, metrics, assumptions, bias, leakage)
- [x] Column quality profile + suspicious flags
- [x] Target characterization + modeling implications
- [x] Prior-only driver ranks (Spearman + MI) with lineage
- [x] Segmentation + temporal stability + descriptive holdout drift
- [x] FE ideas (hypotheses only)
- [x] Modeling-readiness shortlist / avoid list / leakage audit
